<a href="https://colab.research.google.com/github/clarabarretto/Deep-Learning/blob/main/HYAMD_AMD_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reproducao: iBOT para Deteccao de AMD no Dataset HYAMD

Reproduz os experimentos do artigo:
> *Benchmarking Ophthalmology Foundation Models for Clinically Significant Age Macular Degeneration Detection* (Cohen et al., 2025)

**Tarefa:** identificar AMD clinicamente significativo (intermediario a tardio):
> *"we specifically focus on identifying clinically significant AMD, defined as moderate to late stages"*
> *"intermediate-to-late AMD as eyes diagnosed with NVAMD, visible GA, or large drusen (LD)"*

**Binarizacao dos labels (coluna AMD do CSV):**

| AMD no CSV | Significado | Grupo | label |
|---|---|---|---|
| 0 | Controle / DR | non-AMD | 0 |
| 1 | Early AMD | non-AMD | 0 |
| 2 | Intermediario-tardio (NVAMD, GA, LD) | AMD | 1 |

AMD=0 e AMD=1 sao agrupados como non-AMD pois o artigo so considera positivo o AMD moderado a tardio.

**Resultado esperado (Tabela 4 do artigo):**
- iBOT (AREDS -> HYAMD OOD): AUROC `0.806 +/- 0.014`
- AMDNet (multi-source, HYAMD left-out): AUROC `0.842 +/- 0.012`

## 0. Configuracao do Ambiente

In [ ]:
!nvidia-smi

Tue May 26 01:13:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install -q timm==0.9.16 scikit-learn matplotlib pandas pillow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.2 MB/s eta 0:00:00


In [ ]:
import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import timm
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'timm: {timm.__version__}')

Dispositivo: cuda
PyTorch: 2.10.0+cu128
timm: 0.9.16


## 1. Montagem do Google Drive

> Ajuste `HYAMD_ROOT` para o caminho da pasta HYAMD no seu Drive.
>
> Estrutura esperada:
> ```
> HYAMD_ROOT/
>   labels.csv
>   images_per_patient_id.csv
>   images/
>       000917597_L_1.jpg
>       ...
> ```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ======================================================
# AJUSTE ESTE CAMINHO
# ======================================================
HYAMD_ROOT = '/content/drive/MyDrive/hillel-yaffe-fundus-amd/1.0.0'
IMAGES_DIR = f'{HYAMD_ROOT}/Images'
LABELS_CSV = f'{HYAMD_ROOT}/labels/labels.csv'

OUTPUT_DIR = '/content/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Labels : {LABELS_CSV}')
print(f'Imagens: {IMAGES_DIR}')
print(f'Outputs: {OUTPUT_DIR}')

Labels : /content/drive/MyDrive/hillel-yaffe-fundus-amd/1.0.0/labels/labels.csv
Imagens: /content/drive/MyDrive/hillel-yaffe-fundus-amd/1.0.0/Images
Outputs: /content/outputs


## 2. Carregamento e Binarizacao dos Labels

O `labels.csv` tem colunas: `image_id`, `patient_id`, `sex`, `age`, `side`, `AMD`.

**Binarizacao conforme o artigo:**
- `AMD = 0` (controle) → non-AMD → label = 0
- `AMD = 1` (early AMD) → non-AMD → label = 0  ← agrupado com controle
- `AMD = 2` (intermediario-tardio: NVAMD, GA, LD) → AMD → label = 1

In [ ]:
df = pd.read_csv(LABELS_CSV)

print('Shape:', df.shape)
print()
print('Distribuicao original (coluna AMD):')
desc = {0: 'controle/DR', 1: 'early AMD', 2: 'intermediario-tardio (NVAMD/GA/LD)'}
for v, c in df['AMD'].value_counts().sort_index().items():
    print(f'  AMD={v}  {desc[v]:<38}  {c} imagens')

# Binarizacao: somente AMD=2 e positivo
# AMD=0 e AMD=1 sao ambos non-AMD
df['label'] = (df['AMD'] == 2).astype(int)

print()
print('Apos binarizacao (conforme artigo):')
n_pos = int((df['label'] == 1).sum())
n_neg = int((df['label'] == 0).sum())
print(f'  non-AMD (label=0): AMD=0 ({(df["AMD"]==0).sum()}) + AMD=1 ({(df["AMD"]==1).sum()}) = {n_neg} imagens')
print(f'  AMD     (label=1): AMD=2 = {n_pos} imagens')
print(f'  Prevalencia AMD  : {n_pos/(n_pos+n_neg)*100:.1f}%  (artigo reporta 27% para HYAMD)')

# Montar o df com os paths reais das imagens na pasta
images_dir = Path(IMAGES_DIR)

# Mapear image_id -> path real (qualquer extensão)
id_to_path = {f.stem.rstrip('_'): f for f in images_dir.iterdir() if f.is_file()}

df['path'] = df['image_id'].map(id_to_path)

n_missing = df['path'].isna().sum()
print(f'Imagens encontradas: {df["path"].notna().sum()} / {len(df)}')
print(f'Imagens nao encontradas: {n_missing}')

if n_missing > 0:
    print('Exemplos faltando:', df[df['path'].isna()]['image_id'].head(3).tolist())

df = df[df['path'].notna()].copy()

n_pos = int((df['label'] == 1).sum())
n_neg = int((df['label'] == 0).sum())
print(f'  non-AMD (label=0): AMD=0 ({(df["AMD"]==0).sum()}) + AMD=1 ({(df["AMD"]==1).sum()}) = {n_neg} imagens')
print(f'  AMD     (label=1): AMD=2 = {n_pos} imagens')
print(f'  Prevalencia AMD  : {n_pos/(n_pos+n_neg)*100:.1f}%  (artigo reporta 27% para HYAMD)')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/hillel-yaffe-fundus-amd/1.0.0/labels/labels.csv'

## 3. Pre-processamento das Imagens

Implementacao exata da Secao 3 do artigo:
1. **Crop** — remove fundo preto desnecessario
2. **Padding** — torna a imagem quadrada preservando aspect ratio
3. **Resize** — 518x518 com interpolacao bilinear

In [ ]:
def crop_fundus_background(img, threshold=10):
    gray = np.array(img.convert('L'))
    mask = gray > threshold
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any() or not cols.any():
        return img
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    margin = 5
    rmin = max(0, rmin - margin)
    rmax = min(gray.shape[0], rmax + margin)
    cmin = max(0, cmin - margin)
    cmax = min(gray.shape[1], cmax + margin)
    return img.crop((cmin, rmin, cmax, rmax))


def pad_to_square(img, fill=0):
    w, h    = img.size
    max_dim = max(w, h)
    pad_w   = (max_dim - w) // 2
    pad_h   = (max_dim - h) // 2
    padding = (pad_w, pad_h, max_dim - w - pad_w, max_dim - h - pad_h)
    return ImageOps.expand(img, border=padding, fill=fill)


def preprocess_fundus(img, target_size=224):
    img = img.convert('RGB')
    img = crop_fundus_background(img)
    img = pad_to_square(img)
    img = img.resize((target_size, target_size), Image.BILINEAR)
    return img


# Teste visual com um exemplo de cada valor AMD
amd_desc = {0: 'controle (AMD=0)', 1: 'early AMD (AMD=1)', 2: 'AMD mod-tardio (AMD=2)'}
samples  = [df[df['AMD'] == v].sample(1, random_state=SEED).iloc[0] for v in [0, 1, 2]]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for col, row in enumerate(samples):
    img_raw  = Image.open(row['path']).convert('RGB')
    img_proc = preprocess_fundus(img_raw)
    label_str = 'AMD (label=1)' if row['label'] == 1 else 'non-AMD (label=0)'
    axes[0][col].imshow(img_raw)
    axes[0][col].set_title(f"{amd_desc[row['AMD']]}\n{label_str}\nOriginal: {img_raw.size}", fontsize=8)
    axes[0][col].axis('off')
    axes[1][col].imshow(img_proc)
    axes[1][col].set_title(f'Pre-processada: {img_proc.size}', fontsize=8)
    axes[1][col].axis('off')
plt.suptitle('Pre-processamento: crop + pad + resize 224x224', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/preprocessing_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Divisao de Dados por patient_id

Split por paciente: imagens do mesmo paciente nunca aparecem em splits diferentes.

In [ ]:
# 80% pacientes treino+val, 20% teste
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_val_idx, test_idx = next(gss.split(df, groups=df['patient_id']))
train_val_df = df.iloc[train_val_idx].copy()
test_df      = df.iloc[test_idx].copy()

# Dos restantes, 20% validacao
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, val_idx = next(gss2.split(train_val_df, groups=train_val_df['patient_id']))
train_df = train_val_df.iloc[train_idx].copy()
val_df   = train_val_df.iloc[val_idx].copy()

# Garantir ausencia de leak
assert len(set(train_df['patient_id']) & set(val_df['patient_id']))  == 0
assert len(set(train_df['patient_id']) & set(test_df['patient_id'])) == 0
assert len(set(val_df['patient_id'])   & set(test_df['patient_id'])) == 0

print('Split por patient_id -- sem data leakage confirmado')
print()
header = f"{'Split':<12} {'Imgs':>6} {'Pacs':>6} {'AMD':>6} {'non-AMD':>8} {'AMD%':>7}"
print(header)
print('-' * len(header))
for name, sp in [('Treino', train_df), ('Validacao', val_df), ('Teste', test_df)]:
    n     = len(sp)
    n_pat = sp['patient_id'].nunique()
    n_amd = int(sp['label'].sum())
    print(f"{name:<12} {n:>6} {n_pat:>6} {n_amd:>6} {n-n_amd:>8} {n_amd/n*100:>6.1f}%")

for name, sp in [('train', train_df), ('val', val_df), ('test', test_df)]:
    sp.to_csv(f'{OUTPUT_DIR}/split_{name}.csv', index=False)
print('\nSplits salvos.')

## 5. Dataset e DataLoaders

Augmentacoes conforme o artigo:
> *"random horizontal flips and adjustments to contrast, saturation, and hue"*

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE      = 224

train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class FundusDataset(Dataset):
    def __init__(self, dataframe, transform=None, target_size=224):
        self.df          = dataframe.reset_index(drop=True)
        self.transform   = transform
        self.target_size = target_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row['path']).convert('RGB')
        img   = preprocess_fundus(img, target_size=self.target_size)
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return img, label


BATCH_SIZE  = 20
NUM_WORKERS = 2

train_dataset = FundusDataset(train_df, transform=train_transform)
val_dataset   = FundusDataset(val_df,   transform=val_transform)
test_dataset  = FundusDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train : {len(train_dataset):>5} imgs | {len(train_loader):>3} batches')
print(f'Val   : {len(val_dataset):>5} imgs | {len(val_loader):>3} batches')
print(f'Teste : {len(test_dataset):>5} imgs | {len(test_loader):>3} batches')

## 6. Configuração da Arquitetura

Configurar aqui a arquitetura para treino.

In [ ]:
def build_ibot_model(num_classes=1, img_size=224):
    # ViT-Small/16 pré-treinado com DINO (SSL em ImageNet)
    # Mesma família SSL do iBOT, escala reduzida para viabilidade com dataset pequeno
    model = timm.create_model(
        'vit_small_patch16_224.dino',
        pretrained=True,
        num_classes=num_classes,
        img_size=img_size,
        dynamic_img_size=True
    )
    return model


def build_mobilenetv3_large(num_classes=1):
    """MobileNetV3-Large pré-treinado no ImageNet-1k.
    ~5.5 M parâmetros; adequado para inferência em borda."""
    model = timm.create_model(
        'mobilenetv3_large_100',
        pretrained=True,
        num_classes=num_classes
    )
    return model


def build_efficientnet_lite0(num_classes=1):
    """EfficientNet-Lite0 pré-treinado no ImageNet-1k.
    ~4.7 M parâmetros; sem SE-blocks para compatibilidade com hardware restrito."""
    model = timm.create_model(
        'efficientnet_lite0',
        pretrained=True,
        num_classes=num_classes
    )
    return model


def build_convnext_femto(num_classes=1):
    """ConvNeXt-Femto pré-treinado no ImageNet-1k.
    ~5.2 M parâmetros; menor variante da família ConvNeXt no timm."""
    model = timm.create_model(
        'convnext_femto',
        pretrained=True,
        num_classes=num_classes
    )
    return model

model = build_mobilenetv3_large(num_classes=1).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros totais    : {total/1e6:.1f}M')
print(f'Parametros trenaveis : {trainable/1e6:.1f}M')

## 7. Configuracao do Treinamento


In [ ]:
NUM_EPOCHS   = 25
WEIGHT_DECAY = 0.01
PATIENCE = 5
ARCH = "mobilenetv3"
LR           = 5e-4   # dentro do range 1e-5 a 3e-4 do artigo

#    'ibot'              : 1e-4,
#    'mobilenetv3'       : 5e-4,
#    'efficientnet_lite0': 2e-4,
#    'convnext_femto'    : 2e-4,

WARMUP_EPOCHS = 3          # ~10-15% do total de épocas (padrão ConvNeXt fine-tuning)
steps_per_epoch = len(train_loader)
WARMUP_STEPS = WARMUP_EPOCHS * steps_per_epoch  # converte para steps dinamicamente

# Class weighting: N0 * w0 = N1 * w1  =>  pos_weight = N0 / N1
n_neg = int((train_df['label'] == 0).sum())
n_pos = int((train_df['label'] == 1).sum())
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
print(f'Treino -- non-AMD: {n_neg} | AMD: {n_pos} | pos_weight: {pos_weight.item():.3f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
)

total_steps = NUM_EPOCHS * len(train_loader)

def get_scheduler_with_warmup(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scheduler = get_scheduler_with_warmup(optimizer, WARMUP_STEPS, total_steps)
print(f'Total steps: {total_steps} | Warmup: {WARMUP_STEPS} ({WARMUP_STEPS/total_steps*100:.1f}%)')

## 8. Loop de Treinamento

In [ ]:
# ── Funções de avaliação ────────────────────────────────────────────────────

def compute_auroc(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits = model(imgs.to(device)).squeeze(1)
            all_probs.extend(torch.sigmoid(logits).cpu().numpy())
            all_labels.extend(labels.numpy())
    return roc_auc_score(all_labels, all_probs)


def compute_val_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            loss = criterion(model(imgs).squeeze(1), labels)
            total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)


# ── Loop de treino ──────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    for imgs, labels in tqdm(loader, desc='Treino', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs).squeeze(1), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_auroc': []}
best_val_auroc    = 0.0
best_val_loss     = float('inf')
epochs_no_improve = 0
best_model_path   = f'{OUTPUT_DIR}/best_{ARCH}_hyamd.pth'

OVERFIT_RATIO_THRESHOLD = 10.0   # val_loss / train_loss; acima disso = overfitting severo

print(f'Treinando por {NUM_EPOCHS} epochs...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    val_auroc  = compute_auroc(model, val_loader, DEVICE)
    val_loss   = compute_val_loss(model, val_loader, criterion, DEVICE)

    overfit_ratio = val_loss / max(train_loss, 1e-8)

    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auroc'].append(val_auroc)

    # Salva se AUROC melhorou E não há overfitting severo
    auroc_improved = val_auroc > best_val_auroc
    not_overfit    = overfit_ratio < OVERFIT_RATIO_THRESHOLD

    flag = ''
    if auroc_improved and not_overfit:
        best_val_auroc    = val_auroc
        best_val_loss     = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        flag = ' <- melhor'
    else:
        epochs_no_improve += 1

    print(
        f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'Overfit ratio: {overfit_ratio:5.1f}x | '
        f'Val AUROC: {val_auroc:.4f}'
        f'{flag}'
    )

    if epochs_no_improve >= PATIENCE:
        print(f'\nEarly stopping na época {epoch} (sem melhora há {PATIENCE} épocas).')
        break

# Recarrega o melhor checkpoint ao final
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
print(f'\nMelhor Val AUROC : {best_val_auroc:.4f}')
print(f'Val Loss no melhor: {best_val_loss:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['epoch'], history['train_loss'], 'b-o', markersize=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (BCE)')
axes[0].set_title('Training Loss'); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['epoch'], history['val_auroc'], 'g-o', markersize=4)
axes[1].axhline(y=best_val_auroc, color='r', linestyle='--',
                alpha=0.5, label=f'Melhor: {best_val_auroc:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUROC')
axes[1].set_title('Validation AUROC'); axes[1].legend()
axes[1].grid(True, alpha=0.3); axes[1].set_ylim([0.5, 1.0])

plt.suptitle('Curvas de Treinamento -- ViT-S/16 no HYAMD', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
pd.DataFrame(history).to_csv(f'{OUTPUT_DIR}/training_history.csv', index=False)

## 9. Avaliacao Final -- Bootstrap AUROC

Protocolo exato do artigo: AUROC com bootstrap 1.000 amostras usando 80% do conjunto de teste.

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Avaliando no teste'):
        logits = model(imgs.to(DEVICE)).squeeze(1)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)


def bootstrap_auroc(y_true, y_prob, n_bootstrap=1000, sample_frac=0.8, seed=42):
    rng    = np.random.RandomState(seed)
    n      = len(y_true)
    k      = int(n * sample_frac)
    aurocs = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=k, replace=False)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aurocs.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.mean(aurocs), np.std(aurocs)


bs_mean, bs_std = bootstrap_auroc(all_labels, all_probs)
print(f'AUROC Bootstrap (1000x, 80%): {bs_mean:.3f} +/- {bs_std:.3f}')

In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)
fig, axes   = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, 'b-', lw=2,
             label=f'MobileNet-V3-Large (AUROC = {bs_mean:.3f} +/- {bs_std:.3f})')
axes[0].plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
axes[0].fill_between(fpr, tpr, alpha=0.1)
axes[0].set_xlabel('FPR (1 - Especificidade)')
axes[0].set_ylabel('TPR (Sensibilidade)')
axes[0].set_title('Curva ROC -- MobileNet-V3-Large')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].hist(all_probs[all_labels==0], bins=30, alpha=0.6,
             label='non-AMD', color='steelblue', density=True)
axes[1].hist(all_probs[all_labels==1], bins=30, alpha=0.6,
             label='AMD (moderado-tardio)', color='tomato', density=True)
axes[1].axvline(x=0.5, color='k', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Probabilidade prevista de AMD')
axes[1].set_ylabel('Densidade')
axes[1].set_title('Distribuicao de Probabilidades')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Avaliacao Final -- MobileNet-V3-Large  (AUROC = {bs_mean:.3f} +/- {bs_std:.3f})')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roc_curve_and_probs.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Analise de Erros por Subgrupo

Reproducao da Secao 4 do artigo: AUROC por subtipo AMD e por faixa etaria.

In [ ]:
# Salvar predicoes com metadados
results_df = test_df.copy().reset_index(drop=True)
results_df['pred_prob']  = all_probs
results_df['pred_label'] = (all_probs >= 0.5).astype(int)
results_df['correct']    = (results_df['pred_label'] == results_df['label']).astype(int)
results_df.to_csv(f'{OUTPUT_DIR}/test_predictions.csv', index=False)
print('Predicoes salvas em test_predictions.csv')

# AUROC por subtipo: AMD=2 vs todos non-AMD (AMD in {0,1})
print()
print('AUROC por subtipo AMD (vs todos non-AMD = AMD in {0,1}):')
non_amd_mask = results_df['AMD'].isin([0, 1])
mask = (results_df['AMD'] == 2) | non_amd_mask
sub  = results_df[mask].copy()
sub['bin'] = (sub['AMD'] == 2).astype(int)
if sub['bin'].nunique() == 2:
    auc = roc_auc_score(sub['bin'], sub['pred_prob'])
    print(f'  AMD moderado-tardio (AMD=2): AUROC = {auc:.3f}  (n={int(sub["bin"].sum())} casos)')

# AUROC por faixa etaria
print()
print('AUROC por faixa etaria do AMD cohort (AMD=2 vs todos non-AMD):')
age_bins   = [(18, 70), (70, 80), (80, 120)]
age_labels = ['18-70', '70-80', '>80']
auroc_age  = {}
for (lo, hi), lab in zip(age_bins, age_labels):
    in_bin = (results_df['AMD'] == 2) & (results_df['age'] >= lo) & (results_df['age'] < hi)
    subset = results_df[non_amd_mask | in_bin].copy()
    subset['bin'] = in_bin[subset.index].astype(int)
    if subset['bin'].sum() > 0 and subset['bin'].nunique() == 2:
        auc = roc_auc_score(subset['bin'], subset['pred_prob'])
        auroc_age[lab] = auc
        n_a = int(subset['bin'].sum())
        n_c = int((subset['bin']==0).sum())
        print(f'  Idade {lab:>5} | AMD={n_a} vs non-AMD={n_c} | AUROC = {auc:.3f}')
    else:
        print(f'  Idade {lab:>5} | amostras insuficientes')

if auroc_age:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(list(auroc_age.keys()), list(auroc_age.values()),
           color=['#4878CF','#6ACC65','#D65F5F'], width=0.5)
    ax.set_ylim([0.5, 1.0])
    ax.set_xlabel('Faixa Etaria AMD'); ax.set_ylabel('AUROC')
    ax.set_title('AUROC por Faixa Etaria -- similar Figura 2b do artigo')
    ax.grid(True, alpha=0.3, axis='y')
    for i, v in enumerate(auroc_age.values()):
        ax.text(i, v + 0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/auroc_by_age.png', dpi=150, bbox_inches='tight')
    plt.show()

## 11. Grad-CAM -- Mapas de Atencao

Visualizacao dos mapas de atencao conforme a Figura 3 do artigo.

In [ ]:
def overlay_attention(img_pil, cam, alpha=0.6, bg_threshold=10):
    import matplotlib.cm as cm

    # Mascara do fundo preto da imagem original
    img_np   = np.array(img_pil)
    gray     = np.mean(img_np, axis=2)
    fg_mask  = gray > bg_threshold  # True = area do olho, False = fundo preto

    # Redimensionar CAM para o tamanho da imagem
    cam_resized = np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize(img_pil.size, Image.BILINEAR)
    ) / 255.0

    # Normalizar apenas nos pixels do olho (ignorar fundo)
    cam_fg = cam_resized.copy()
    if fg_mask.any():
        vmin = cam_resized[fg_mask].min()
        vmax = cam_resized[fg_mask].max()
        if vmax > vmin:
            cam_fg[fg_mask]  = (cam_resized[fg_mask] - vmin) / (vmax - vmin)
        cam_fg[~fg_mask] = 0.0  # fundo recebe valor 0 mas sera mascarado

    # Aplicar colormap
    heatmap = cm.jet(cam_fg)[:, :, :3]

    # Composicao: fundo permanece preto, olho recebe overlay
    img_float  = img_np / 255.0
    overlay    = img_float.copy()
    overlay[fg_mask] = (
        alpha * heatmap[fg_mask] + (1 - alpha) * img_float[fg_mask]
    )
    # Fundo permanece a imagem original (preto)
    return Image.fromarray((overlay * 255).astype(np.uint8))

# 1 exemplo de cada classe
attn_cam = AttentionCAM_ViT(model)
samples  = pd.concat([
    results_df[results_df['label'] == 0].sample(1, random_state=SEED),
    results_df[results_df['label'] == 1].sample(1, random_state=SEED),
]).reset_index(drop=True)

label_names = {0: 'non-AMD (label=0)', 1: 'AMD moderado-tardio (label=1)'}

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for col, (_, row) in enumerate(samples.iterrows()):
    img_pil    = preprocess_fundus(Image.open(row['path']).convert('RGB'))
    img_tensor = val_transform(img_pil).unsqueeze(0).to(DEVICE)
    cam        = attn_cam.generate(img_tensor)
    with torch.no_grad():
        prob = torch.sigmoid(model(img_tensor)).item()
    overlay = overlay_attention(img_pil, cam, alpha=0.6, bg_threshold=10)

    axes[0][col].imshow(img_pil)
    axes[0][col].set_title(f"{label_names[row['label']]}\np={prob:.2f}", fontsize=9)
    axes[0][col].axis('off')

    axes[1][col].imshow(overlay)
    axes[1][col].set_title('Attention Map', fontsize=9)
    axes[1][col].axis('off')

plt.suptitle('Mapas de Atencao -- ViT-S/16 DINO no HYAMD', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Grad-CAM para CNNs (MobileNetV3, EfficientNet-Lite0, ConvNeXt-Femto) ────
ARCH = "mobilenetv3"

def overlay_attention(img_pil, cam, alpha=0.6, bg_threshold=10):
    import matplotlib.cm as cm

    img_np  = np.array(img_pil)
    gray    = np.mean(img_np, axis=2)
    fg_mask = gray > bg_threshold

    cam_resized = np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize(img_pil.size, Image.BILINEAR)
    ) / 255.0

    cam_fg = cam_resized.copy()
    if fg_mask.any():
        vmin = cam_resized[fg_mask].min()
        vmax = cam_resized[fg_mask].max()
        if vmax > vmin:
            cam_fg[fg_mask] = (cam_resized[fg_mask] - vmin) / (vmax - vmin)
        cam_fg[~fg_mask] = 0.0

    heatmap   = cm.jet(cam_fg)[:, :, :3]
    img_float = img_np / 255.0
    overlay   = img_float.copy()
    overlay[fg_mask] = alpha * heatmap[fg_mask] + (1 - alpha) * img_float[fg_mask]
    return Image.fromarray((overlay * 255).astype(np.uint8))

class GradCAM_CNN:
    """
    Grad-CAM genérico para CNNs do timm.
    Hookeia automaticamente a última camada convolucional de cada arquitetura.
    """

    # Mapeamento: nome do ARCH -> atributo do modelo onde está a última conv
    _LAYER_MAP = {
        'mobilenetv3'       : 'blocks',      # último bloco de features
        'efficientnet_lite0': 'blocks',
        'convnext_femto'    : 'stages',
    }

    def __init__(self, model, arch=ARCH):
        self.model    = model
        self.arch     = arch
        self.gradients = None
        self.activations = None
        self._hook_handles = []
        self._register_hooks()

    def _get_target_layer(self):
        """Retorna a última camada convolucional/bloco de features do modelo."""
        if self.arch == 'mobilenetv3':
            # Último InvertedResidual antes do pooling global
            return self.model.blocks[-1]
        elif self.arch == 'efficientnet_lite0':
            return self.model.blocks[-1]
        elif self.arch == 'convnext_femto':
            # Último stage do ConvNeXt
            return self.model.stages[-1]
        else:
            raise ValueError(f"Arch '{self.arch}' não mapeada no GradCAM_CNN. "
                             f"Opções: {list(self._LAYER_MAP.keys())}")

    def _register_hooks(self):
        layer = self._get_target_layer()

        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self._hook_handles.append(layer.register_forward_hook(forward_hook))
        self._hook_handles.append(layer.register_full_backward_hook(backward_hook))

    def remove_hooks(self):
        for h in self._hook_handles:
            h.remove()

    def generate(self, img_tensor):
        """
        Retorna o mapa Grad-CAM normalizado [0, 1] com shape (H', W').
        img_tensor: (1, C, H, W) já no device correto.
        """
        self.model.eval()
        self.model.zero_grad()

        # Forward com gradientes habilitados
        with torch.enable_grad():
            img_tensor = img_tensor.requires_grad_(False)
            logit = self.model(img_tensor).squeeze()
            # Grad-CAM em relação à saída (classificação binária → scalar direto)
            logit.backward()

        # Gradientes: (1, C, H, W) → média espacial → pesos por canal
        grads = self.gradients                  # (1, C, H', W')
        acts  = self.activations                # (1, C, H', W')

        # Global Average Pooling dos gradientes (pesos Grad-CAM)
        weights = grads.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)

        # Combinação linear ponderada + ReLU
        cam = (weights * acts).sum(dim=1).squeeze()     # (H', W')
        cam = torch.relu(cam)

        # Normalização para [0, 1]
        cam = cam.cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)

        return cam


# ── Visualização (overlay_attention reutilizada do ViT, sem alteração) ───────

# Instanciar o Grad-CAM para o modelo CNN atual
grad_cam = GradCAM_CNN(model, arch=ARCH)

samples = pd.concat([
    results_df[results_df['label'] == 0].sample(1, random_state=SEED),
    results_df[results_df['label'] == 1].sample(1, random_state=SEED),
]).reset_index(drop=True)

label_names = {0: 'non-AMD (label=0)', 1: 'AMD moderado-tardio (label=1)'}

fig, axes = plt.subplots(2, 2, figsize=(8, 8))

for col, (_, row) in enumerate(samples.iterrows()):
    img_pil    = preprocess_fundus(Image.open(row['path']).convert('RGB'))
    img_tensor = val_transform(img_pil).unsqueeze(0).to(DEVICE)

    cam = grad_cam.generate(img_tensor)

    with torch.no_grad():
        prob = torch.sigmoid(model(img_tensor)).item()

    overlay = overlay_attention(img_pil, cam, alpha=0.6, bg_threshold=10)

    axes[0][col].imshow(img_pil)
    axes[0][col].set_title(f"{label_names[row['label']]}\np={prob:.2f}", fontsize=9)
    axes[0][col].axis('off')

    axes[1][col].imshow(overlay)
    axes[1][col].set_title('Grad-CAM', fontsize=9)
    axes[1][col].axis('off')

arch_label = ARCH.replace('_', ' ').title()
plt.suptitle(f'Grad-CAM — {arch_label} no HYAMD', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/gradcam_{ARCH}.png', dpi=150, bbox_inches='tight')
plt.show()

# Remover hooks ao final para não interferir em chamadas subsequentes
grad_cam.remove_hooks()

## 12. Resumo Final

In [ ]:
summary = {
    'dataset'                  : 'HYAMD',
    'modelo'                   : 'MobileNet-V3-Large',
    'binarizacao'              : 'AMD=2 -> label=1 | AMD in {0,1} -> label=0',
    'split_por'                : 'patient_id (sem data leakage)',
    'n_train_imgs'             : int(len(train_df)),
    'n_val_imgs'               : int(len(val_df)),
    'n_test_imgs'              : int(len(test_df)),
    'auroc_bootstrap_mean'     : round(float(bs_mean), 3),
    'auroc_bootstrap_std'      : round(float(bs_std),  3),
    'epochs'                   : NUM_EPOCHS,
    'batch_size'               : BATCH_SIZE,
    'lr'                       : LR,
    'weight_decay'             : WEIGHT_DECAY,
    'warmup_steps'             : WARMUP_STEPS,
    'img_size'                 : IMG_SIZE,
}

with open(f'{OUTPUT_DIR}/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('=' * 55)
print('         RESUMO FINAL DOS RESULTADOS')
print('=' * 55)
for k, v in summary.items():
    print(f'  {k:<35} {v}')
print('=' * 55)
print(f'\nArquivos salvos em {OUTPUT_DIR}:')
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f'  {f.name}')